# Generate danish model donuts from butler information

Owner: **Bryce Kalmbach** ([@jbkalmbach](https://github.com/lsst-sitcom/ts_aos_analysis/issues/new?body=@jbkalmbach)) 

Last Verified to Run: **2025-11-11**

Software Versions:
* ts_wep: **v15.0.2 or later**
* donut_viz: **v2.5.0 or later**

The following notebook shows how to generate the danish model using information stored in the butler and the `donut_viz` package.

## Imports

All that's needed from `donut_viz` now is the `PlotDonutFitsTask`. All other information is already in the butler in the `aggregateAOSVisitTableRaw`, `donutStampsIntraVisit`, and `donutStampExtraVisit` data products.

This requires `donut_viz` version v2.5.0 or later.

In [ ]:
from lsst.daf.butler import Butler
from matplotlib import pyplot as plt
from lsst.donut.viz import PlotDonutFitsTask
%matplotlib inline

## Get data

All data in the `u/brycek/aos_cwfs_danish` collection with a `day_obs >= 20251020` has been run with at least a `ts_wep` version equal to or later than v15.0.2 that added danish parameters to the `aggregateAOSVisitTableRaw` table metadata.

In [ ]:
collection_name = 'aos_cwfs_danish'
butler = Butler('main', collections=collection_name, instrument='LSSTCam')

In [ ]:
agg_raw_list = butler.query_datasets(
    'aggregateAOSVisitTableRaw', 
    where="exposure.day_obs >= 20251020"
)

In [ ]:
# Pick a visit from the list
visit_idx = 55
agg_raw = butler.get(agg_raw_list[visit_idx])

In [ ]:
# The following keys are necessary in "estimatorInfo"
agg_raw.meta['estimatorInfo'].keys()

In [ ]:
intra_stamps = butler.get('donutStampsIntraVisit', dataId=agg_raw_list[visit_idx].dataId)
extra_stamps = butler.get('donutStampsExtraVisit', dataId=agg_raw_list[visit_idx].dataId)

In [ ]:
# Pick a stamp from the visit
stamp_idx = 2
danish_meta = {key: value[stamp_idx] for key, value in agg_raw.meta['estimatorInfo'].items()}
noll_indices = agg_raw.meta['nollIndices']

In [ ]:
# The danish components that are provided for the model
danish_meta

## Run `PlotDonutFitsTask.getModel` to create the model and compare to the real data

In [ ]:
plotDonutFits = PlotDonutFitsTask()

In [ ]:
imgs, model_imgs = plotDonutFits.getModel(agg_raw[stamp_idx]['zk_CCS'], noll_indices, danish_meta, extra_stamps[stamp_idx], intra_stamps[stamp_idx])

In [ ]:
fig = plt.figure(figsize=(8, 8))

fig.add_subplot(2,2,1)
plt.imshow(imgs[0], origin='lower')
plt.title('Extra-Focal Image')
fig.add_subplot(2,2,2)
plt.imshow(model_imgs[0], origin='lower')
plt.title('Extra-Focal Model')

fig.add_subplot(2,2,3)
plt.imshow(imgs[1], origin='lower')
plt.title('Intra-Focal Image')
fig.add_subplot(2,2,4)
plt.imshow(model_imgs[1], origin='lower')
plt.title('Intra-Focal Model')